# 01 — Nettoyage & préparation des données
## Projet : Coût de la vie et inflation au Sénégal (2018–2026)

**Objectif du notebook** : transformer les données *brutes* (`data/raw/`) en un
modèle en étoile propre et enrichi (`data/processed/`) prêt pour l'analyse et
Power BI.

### ⚠️ Note de transparence sur les sources
- **Agrégats nationaux** (taux d'inflation annuel, pic de **+9,7 %** en 2022 avec
  un sommet de **+14,1 %** en novembre 2022, désinflation à **+5,9 %** en 2023 et
  **+0,8 %** en 2024, base **100 = 2023**, 12 divisions COICOP/NCOA, 6 zones de
  collecte) : **calibrés sur les chiffres officiels ANSD / Banque mondiale**.
- **Détail mensuel par produit et par région** : **reconstruit** de façon
  cohérente avec ces agrégats (l'ANSD ne publie pas cette granularité en format
  structuré). À ne pas citer comme chiffre officiel produit par produit.

### Opérations de nettoyage réalisées
1. Harmonisation des **formats de dates** (3 formats mélangés).
2. Uniformisation de la **casse des régions**.
3. Correction du **séparateur décimal** (virgule → point) et typage numérique.
4. Suppression des **doublons**.
5. Traitement des **valeurs aberrantes** (erreurs de saisie ×10).
6. Imputation des **valeurs manquantes** (interpolation temporelle).
7. **Variables dérivées** : variation mensuelle, glissement annuel, moyenne
   mobile, indice base 100, prix réel déflaté, pouvoir d'achat.


In [ ]:

import os, warnings, pathlib
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.figsize"] = (11, 5)
plt.rcParams["axes.titlesize"] = 13
plt.rcParams["figure.dpi"] = 110

PROJ = os.getcwd()
if not os.path.isdir(os.path.join(PROJ, "data")):
    PROJ = os.path.dirname(PROJ)
RAW = os.path.join(PROJ, "data", "raw")
PROC = os.path.join(PROJ, "data", "processed")
FIG = os.path.join(PROJ, "reports", "figures")
MODELS = os.path.join(PROJ, "models")
for d in (PROC, FIG, MODELS):
    os.makedirs(d, exist_ok=True)
print("Racine projet :", PROJ)


### 1. Chargement des données brutes

In [ ]:

prix_raw = pd.read_csv(os.path.join(RAW, "prix_produits_brut.csv"), dtype=str)
ihpc_raw = pd.read_csv(os.path.join(RAW, "ihpc_regions_brut.csv"), dtype=str)
ref_regions = pd.read_csv(os.path.join(RAW, "ref_regions.csv"))
ref_div = pd.read_csv(os.path.join(RAW, "ref_divisions.csv"))
ref_prod = pd.read_csv(os.path.join(RAW, "ref_produits.csv"))

print("Prix bruts   :", prix_raw.shape)
print("IHPC bruts   :", ihpc_raw.shape)
prix_raw.head()


### 2. Diagnostic qualité (avant nettoyage)

In [ ]:

print("Valeurs manquantes (prix) :\n", prix_raw.isna().sum(), "\n")
print("Doublons exacts (prix)    :", prix_raw.duplicated().sum())
print("\nExemples de formats de dates rencontrés :")
print(prix_raw["date"].drop_duplicates().head(6).tolist())
print("\nExemples de libellés de régions :")
print(sorted(prix_raw["region"].unique())[:12])
print("\nExemples de prix (texte brut) :")
print(prix_raw["prix_moyen"].dropna().head(6).tolist())


### 3. Fonctions de nettoyage

In [ ]:

import re
MOIS_FR = {"janvier":1,"février":2,"mars":3,"avril":4,"mai":5,"juin":6,
           "juillet":7,"août":8,"septembre":9,"octobre":10,"novembre":11,"décembre":12}

def parse_date(s):
    s = str(s).strip()
    if re.match(r"^\d{4}-\d{2}-\d{2}$", s):
        return pd.Timestamp(s)
    if re.match(r"^\d{2}/\d{2}/\d{4}$", s):
        d, m, y = s.split("/"); return pd.Timestamp(int(y), int(m), int(d))
    parts = s.split()
    if len(parts) == 2 and parts[0].lower() in MOIS_FR:
        return pd.Timestamp(int(parts[1]), MOIS_FR[parts[0].lower()], 1)
    return pd.NaT

def clean_region(s):
    return str(s).strip().title()

def clean_price(v):
    if v is None or (isinstance(v, float) and pd.isna(v)):
        return np.nan
    s = str(v).strip()
    if s == "" or s.lower() == "nan":
        return np.nan
    return float(s.replace(",", "."))

# Vérification rapide
assert parse_date("2020-03-01") == pd.Timestamp("2020-03-01")
assert parse_date("01/03/2020") == pd.Timestamp("2020-03-01")
assert parse_date("Mars 2020") == pd.Timestamp("2020-03-01")
assert clean_price("1 250,5".replace(" ", "")) == 1250.5
print("Fonctions de nettoyage OK")


### 4. Application du nettoyage

In [ ]:

def nettoyer(df, valcol):
    df = df.copy()
    df["date"] = df["date"].map(parse_date)
    df["region"] = df["region"].map(clean_region)
    df[valcol] = df[valcol].map(clean_price)
    # normalisation de tous les jours au 1er du mois
    df["date"] = df["date"].values.astype("datetime64[M]")
    return df

prix = nettoyer(prix_raw, "prix_moyen")
ihpc = nettoyer(ihpc_raw, "indice")
ihpc["ponderation"] = pd.to_numeric(ihpc["ponderation"], errors="coerce")

print("Régions après nettoyage :", sorted(prix["region"].unique()))
print("Période :", prix["date"].min().date(), "->", prix["date"].max().date())


### 5. Doublons & valeurs aberrantes

In [ ]:

# Doublons : on garde une ligne par (date, region, produit)
n0 = len(prix)
prix = (prix.sort_values("date")
            .drop_duplicates(subset=["date", "region", "produit"], keep="first"))
print(f"Doublons supprimés : {n0 - len(prix)}")

# Valeurs aberrantes : prix hors [0,2 ; 5] × médiane du produit -> NaN
med = prix.groupby("produit")["prix_moyen"].transform("median")
outliers = (prix["prix_moyen"] > 5 * med) | (prix["prix_moyen"] < 0.2 * med)
print(f"Valeurs aberrantes neutralisées : {int(outliers.sum())}")
prix.loc[outliers, "prix_moyen"] = np.nan


### 6. Imputation des valeurs manquantes (interpolation temporelle)

In [ ]:

def imputer(df, valcol, keys):
    df = df.sort_values(keys + ["date"]).copy()
    df[valcol] = (df.groupby(keys)[valcol]
                    .transform(lambda s: s.interpolate(limit_direction="both")
                                          .ffill().bfill()))
    return df

prix = imputer(prix, "prix_moyen", ["produit", "region"])
ihpc = imputer(ihpc, "indice", ["division_code", "region"])
print("NaN restants prix :", int(prix["prix_moyen"].isna().sum()))
print("NaN restants ihpc :", int(ihpc["indice"].isna().sum()))


### 7. Variables dérivées
Variation mensuelle, **glissement annuel** (variation vs même mois N-1),
**moyenne mobile 3 mois**.

In [ ]:

def ajouter_variations(df, valcol, keys, prefix):
    df = df.sort_values(keys + ["date"]).copy()
    g = df.groupby(keys)[valcol]
    df[f"{prefix}_var_mensuelle_pct"] = g.pct_change(1) * 100
    df[f"{prefix}_var_annuelle_pct"] = g.pct_change(12) * 100
    df[f"{prefix}_ma3"] = g.transform(lambda s: s.rolling(3, min_periods=1).mean())
    return df

prix = ajouter_variations(prix, "prix_moyen", ["produit", "region"], "prix")
ihpc = ajouter_variations(ihpc, "indice", ["division_code", "region"], "ihpc")
prix.head(3)


### 8. Reconstruction de l'IHPC national et des sous-indices
- **IHPC national** = moyenne des indices régionaux (pondérée par le poids
  démographique des zones), puis agrégation pondérée des 12 divisions.
- **Indice alimentaire** = division 01.
- **Indice énergie** = construit à partir des produits énergétiques administrés
  (gaz, essence, gasoil, électricité).

In [ ]:

# Poids démographiques approximatifs des 6 zones de collecte
POIDS_ZONE = {"Dakar":0.34, "Thiès":0.18, "Diourbel":0.14,
              "Kaolack":0.12, "Saint-Louis":0.12, "Kolda":0.10}
ihpc["poids_zone"] = ihpc["region"].map(POIDS_ZONE)

# Indice national par division (moyenne pondérée des régions)
nat_div = (ihpc.assign(w=ihpc["poids_zone"])
               .groupby(["date", "division_code", "division"])
               .apply(lambda g: np.average(g["indice"], weights=g["w"]))
               .reset_index(name="indice"))
# poids des divisions
poids_div = ref_div.set_index("code")["poids_frac"].to_dict()
nat_div["poids_frac"] = nat_div["division_code"].map(poids_div)

# IHPC global national
ihpc_nat = (nat_div.assign(p=nat_div["indice"] * nat_div["poids_frac"])
                   .groupby("date")["p"].sum().reset_index(name="indice_global"))
ihpc_nat = ihpc_nat.sort_values("date")
ihpc_nat["var_mensuelle_pct"] = ihpc_nat["indice_global"].pct_change(1) * 100
ihpc_nat["var_annuelle_pct"] = ihpc_nat["indice_global"].pct_change(12) * 100

# Indice alimentaire (division 01)
alim = nat_div[nat_div["division_code"] == "D01"][["date", "indice"]].rename(
    columns={"indice": "indice_alimentaire"})
ihpc_nat = ihpc_nat.merge(alim, on="date", how="left")
ihpc_nat["var_alim_annuelle_pct"] = ihpc_nat["indice_alimentaire"].pct_change(12) * 100

# Indice énergie à partir des prix administrés (base 100 = 2023, national = moyenne régions)
ENER = ["Gaz butane (6 kg)", "Essence super (SP95)", "Gasoil (diesel)", "Électricité (tranche sociale)"]
ener = prix[prix["produit"].isin(ENER)].copy()
ener_nat = ener.groupby(["date", "produit"])["prix_moyen"].mean().reset_index()
base2023 = (ener_nat[ener_nat["date"].dt.year == 2023]
            .groupby("produit")["prix_moyen"].mean())
ener_nat["rel"] = ener_nat.apply(lambda r: r["prix_moyen"] / base2023[r["produit"]] * 100, axis=1)
indice_energie = ener_nat.groupby("date")["rel"].mean().reset_index(name="indice_energie")
ihpc_nat = ihpc_nat.merge(indice_energie, on="date", how="left")
ihpc_nat["var_energie_annuelle_pct"] = ihpc_nat["indice_energie"].pct_change(12) * 100

# Pouvoir d'achat (base 2023 = 100) : inverse de l'indice global
ihpc_nat["pouvoir_achat_index"] = 100 * (100 / ihpc_nat["indice_global"])

ihpc_nat.tail(6)[["date","indice_global","var_annuelle_pct","var_alim_annuelle_pct","var_energie_annuelle_pct","pouvoir_achat_index"]]


### 9. Validation : inflation reconstruite vs chiffres officiels ANSD

In [ ]:

v = ihpc_nat.dropna(subset=["var_annuelle_pct"]).copy()
v["annee"] = v["date"].dt.year
recon = v.groupby("annee")["var_annuelle_pct"].mean().round(2)
officiel = {2019:1.0, 2020:2.5, 2021:2.2, 2022:9.7, 2023:5.9, 2024:0.8, 2025:2.0}
comp = pd.DataFrame({"reconstruit_%": recon,
                     "officiel_ANSD_%": pd.Series(officiel)}).dropna()
comp["écart_pt"] = (comp["reconstruit_%"] - comp["officiel_ANSD_%"]).round(2)
print(comp)
print("\nPic de glissement annuel :",
      round(ihpc_nat["var_annuelle_pct"].max(), 1), "% en",
      ihpc_nat.loc[ihpc_nat["var_annuelle_pct"].idxmax(), "date"].strftime("%B %Y"))


### 10. IHPC régional + prix réel déflaté

In [ ]:

# IHPC global par région (agrégation des divisions au sein de chaque région)
ihpc_reg = (ihpc.assign(p=ihpc["indice"] * ihpc["division_code"].map(poids_div))
                .groupby(["date", "region", "zone"])["p"].sum()
                .reset_index(name="indice_global"))
ihpc_reg = ihpc_reg.sort_values(["region", "date"])
ihpc_reg["var_annuelle_pct"] = (ihpc_reg.groupby("region")["indice_global"]
                                .pct_change(12) * 100)

# Prix réel : déflaté par l'IHPC national (FCFA constants 2023)
defl = ihpc_nat.set_index("date")["indice_global"]
prix = prix.merge(defl.rename("ihpc_defl"), left_on="date", right_index=True, how="left")
prix["prix_reel"] = prix["prix_moyen"] / prix["ihpc_defl"] * 100
prix = prix.drop(columns=["ihpc_defl"])
ihpc_reg.head(3)


### 11. Coût du panier de base
Coût mensuel d'un panier-type de consommation (quantités définies dans le
référentiel produits), nominal et réel (déflaté).

In [ ]:

qte = ref_prod.set_index("produit")["qte_panier"].to_dict()
panier = prix.copy()
panier["qte"] = panier["produit"].map(qte)
panier["cout"] = panier["prix_moyen"] * panier["qte"]
panier_reg = panier.groupby(["date", "region"])["cout"].sum().reset_index(name="cout_panier")
# National = moyenne pondérée par poids de zone
panier_reg["w"] = panier_reg["region"].map(POIDS_ZONE)
panier_nat = (panier_reg.groupby("date")
              .apply(lambda g: np.average(g["cout_panier"], weights=g["w"]))
              .reset_index(name="cout_panier"))
panier_nat = panier_nat.merge(defl.rename("ihpc"), left_on="date", right_index=True)
panier_nat["cout_panier_reel"] = panier_nat["cout_panier"] / panier_nat["ihpc"] * 100
panier_reg = panier_reg.drop(columns=["w"])
print("Coût panier national (FCFA/mois) - début vs fin :")
print(round(panier_nat["cout_panier"].iloc[0]), "->", round(panier_nat["cout_panier"].iloc[-1]))


### 12. Construction du modèle en étoile et écriture dans `data/processed/`

In [ ]:

# --- Dimensions ---
dim_date = pd.DataFrame({"date": sorted(ihpc_nat["date"].unique())})
dim_date["annee"] = dim_date["date"].dt.year
dim_date["mois"] = dim_date["date"].dt.month
mois_noms = ["Janvier","Février","Mars","Avril","Mai","Juin","Juillet","Août",
             "Septembre","Octobre","Novembre","Décembre"]
dim_date["mois_nom"] = dim_date["mois"].map(lambda m: mois_noms[m-1])
dim_date["trimestre"] = "T" + dim_date["date"].dt.quarter.astype(str)
dim_date["annee_mois"] = dim_date["date"].dt.strftime("%Y-%m")

dim_region = ref_regions.copy()
dim_region["poids_zone"] = dim_region["region"].map(POIDS_ZONE)
dim_division = ref_div.rename(columns={"code":"division_code", "libelle":"division"})
dim_produit = ref_prod.rename(columns={"division_code":"division_code"})

# --- Faits ---
fact_ihpc = ihpc.rename(columns={
    "ihpc_var_mensuelle_pct":"var_mensuelle_pct",
    "ihpc_var_annuelle_pct":"var_annuelle_pct",
    "ihpc_ma3":"indice_ma3"})[
    ["date","region","zone","division_code","division","ponderation","indice",
     "var_mensuelle_pct","var_annuelle_pct","indice_ma3"]]

fact_prix = prix.rename(columns={
    "prix_var_mensuelle_pct":"var_mensuelle_pct",
    "prix_var_annuelle_pct":"var_annuelle_pct",
    "prix_ma3":"prix_ma3"})[
    ["date","region","produit","division_code","categorie","unite","prix_moyen",
     "var_mensuelle_pct","var_annuelle_pct","prix_ma3","prix_reel"]]

tables = {
    "dim_date": dim_date, "dim_region": dim_region, "dim_division": dim_division,
    "dim_produit": dim_produit, "fact_ihpc": fact_ihpc, "fact_prix": fact_prix,
    "ihpc_national": ihpc_nat, "ihpc_regional": ihpc_reg,
    "panier_national": panier_nat, "panier_regional": panier_reg,
}
for name, df in tables.items():
    df.to_csv(os.path.join(PROC, f"{name}.csv"), index=False, encoding="utf-8-sig")
    print(f"  {name:18s} -> {df.shape}")
print("\n✅ data/processed/ généré : modèle en étoile prêt pour l'analyse et Power BI.")
